In [2]:
import pandas as pd
import os

In [ ]:
directory = r'..\EvolveTest'
dms = 'DMS_tests.xlsx'
dms_path = os.path.join(directory,dms)
seq_names = "variant"
dataset =  'HA-H1N1'
exp_name ="doud"
renaming = {seq_names : "seq_id", exp_name: "activity"}
renamed_xlsx = os.path.join(directory,'DMS_data',(dataset+"_"+exp_name+"_DMS.xlsx"))
df = pd.read_excel(dms_path,sheet_name=dataset)
df = df[[seq_names,exp_name]]
df =df.rename(columns=renaming)
df = df.dropna()\
    .to_excel(renamed_xlsx,index=False)
display(df)

None

In [5]:
import os
import pandas as pd
directory = r'..\EvolveTest\DMS_data\DMS_ProteinGym_substitutions'

folder_path = directory
seq_names = "mutant"
exp_name ="DMS_score"
renaming = {seq_names : "seq_id", exp_name: "activity"}
def reorder_mutation_labels(df, column_name):
    def reorder_label(label):
        if '_' not in label:
            return label
            
        parts = label.split('_')
        if len(parts) != 2:
            return label
            
        # Extract numbers from each part
        import re
        num1 = int(re.search(r'\d+', parts[0]).group())
        num2 = int(re.search(r'\d+', parts[1]).group())
        
        # If already in order, return original
        if num1 <= num2:
            return label
            
        # Otherwise swap
        return parts[1] + '_' + parts[0]
    
    # Apply the function to the specified column
    df[column_name] = df[column_name].apply(reorder_label)
    
    return df
df = pd.read_excel(r'../EvolveTest/DMS_data/DMS_ProteinGym_substitutions/SPG1_STRSG_Olson_2014.xlsx')
print(df)
df = reorder_mutation_labels(df,'seq_id')
df.to_excel(r'../EvolveTest/DMS_data/DMS_ProteinGym_substitutions/SPG1_STRSG_Olson_2014(2).xlsx',index=False)
#for filename in os.listdir(folder_path):
#    print(filename)
#    file_path = os.path.join(folder_path, filename)
#    if os.path.isfile(file_path):  # Check if it's a file, not a subfolder
#        df = pd.read_csv(file_path)
#        df = df[[seq_names,exp_name]]
#        df =df.rename(columns=renaming)
#        df['seq_id'] = df['seq_id'].apply(lambda x: x.replace(":","_"))
#        df = df.dropna()\
#            .to_excel(os.path.join(folder_path,(filename[:-3]+'xlsx')),index=False)



#dms_path = os.path.join(directory,dms)

#dataset = 'zika'

#renamed_xlsx = os.path.join(directory,'DMS_data',(dataset+
#                                        "_"+exp_name+
#                                        "_dms.xlsx"))
##df = pd.read_csv(dms_path)
#df = pd.read_excel(dms_path,
#                   #sheet_name='MAPK1'
#                   )
#df = df[[seq_names,exp_name]]
#df =df.rename(columns=renaming)
#df = df.dropna()\
#    .to_excel(renamed_xlsx,index=False)
#display(df)

             seq_id  activity
0       Q228R_T279L  0.328504
1       Q228K_T279A  0.702656
2       Q228K_T279C  0.422857
3       Q228K_T279D  0.263417
4       Q228K_T279E  0.734820
...             ...       ...
536957  E282Q_T251Q  0.202574
536958  E282Q_T251P -0.289294
536959  E282Q_T251N -0.688549
536960  E282Q_T270E -0.916291
536961  E282Y_T242K -1.201266

[536962 rows x 2 columns]


In [12]:
import os
import pandas as pd
directory = r'..\EvolveTest'
dms = 'DMS_data'
dms_path = os.path.join(directory,dms)
seq_names = "variant"
dataset = 'amiE'
exp_name ="propi"
renamed_xlsx = os.path.join(directory,'DMS_data',(dataset+
                                        "_"+exp_name+
                                        "_dms.xlsx"))
#df = pd.read_csv(dms_path)
df = pd.read_excel(renamed_xlsx)
df['activity'] = df['activity'] + 1
df.to_excel(renamed_xlsx,index=False)
print(df)

     seq_id  activity
0       S9I    1.6352
1       S9C    1.6272
2       H3I    1.6016
3       S9H    1.5562
4       H3V    1.5497
...     ...       ...
6286  C139E   -3.9792
6287  F223E   -4.7994
6288  C178L   -5.6954
6289  N219E   -9.0000
6290  A221F   -9.0000

[6291 rows x 2 columns]


In [4]:
!pip install plotly

   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   ---------------------------------------- 14.8/14.8 MB 93.2 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
# or alternatively with matplotlib
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

# Assuming your dataframe is called 'df' with columns ['variant', 'fitness']

def parse_mutation(variant):
    """Parse mutation string like 'A26S' into components"""
    wt = variant[0]    # wild-type amino acid
    pos = int(variant[1:-1])  # position
    mut = variant[-1]  # mutant amino acid
    return wt, pos, mut
activity = pd.read_excel(r'..\EvolveTest\DMS_data\rubisco_dms.xlsx')
embeddings = pd.read_csv(r'..\EvolveTest\rubisco\embeddings\rubisco_esmc_600m_WT.csv')
# Create new columns for position and mutant amino acid
# Create new columns for position and mutant amino acid
df = pd.merge(activity,embeddings, on='seq_id')
df[['wt', 'position', 'mutation']] = df['seq_id'].apply(lambda x: pd.Series(parse_mutation(x)))
pivot_data = df.pivot(index='position', columns='mutation', values='activity')

# Create meshgrid
X, Y = np.meshgrid(np.arange(len(pivot_data.columns)), np.arange(len(pivot_data.index)))

# Calculate threshold for top 10%
threshold = np.percentile(pivot_data.values[~np.isnan(pivot_data.values)], 90)

# Create main surface
fig = go.Figure()

# Add main surface (bottom 90%)
fig.add_trace(go.Surface(
    z=np.where(pivot_data.values < threshold, pivot_data.values, threshold),
    x=X,
    y=Y,
    colorscale='viridis',
    showscale=True,
    name='Fitness'
))

# Add highlighted surface (top 10%)
fig.add_trace(go.Surface(
    z=np.where(pivot_data.values >= threshold, pivot_data.values, np.nan),
    x=X,
    y=Y,
    colorscale=[[0, 'red'], [1, 'red']],  # Highlight color
    showscale=False,
    opacity=0.8,
    name='Top 10%'
))

# Update layout to remove axis labels and clean up appearance
fig.update_layout(
    scene = dict(
        xaxis = dict(showticklabels=False, showgrid=False, zeroline=False, showline=False),
        yaxis = dict(showticklabels=False, showgrid=False, zeroline=False, showline=False),
        zaxis = dict(title='Fitness'),
    ),
    width=1000,
    height=800,
    showlegend=False
)

fig.show()

In [12]:

import numpy as np
import plotly.graph_objects as go
# or alternatively with matplotlib
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import ast


# Assuming your dataframe is called 'df' with columns ['variant', 'fitness']

def parse_mutation(variant):
    """Parse mutation string like 'A26S' into components"""
    wt = variant[0]    # wild-type amino acid
    pos = int(variant[1:-1])  # position
    mut = variant[-1]  # mutant amino acid
    return wt, pos, mut
activity = pd.read_excel(r'..\EvolveTest\DMS_data\rubisco_dms.xlsx')
embeddings = pd.read_csv(r'..\EvolveTest\rubisco\embeddings\rubisco_esmc_600m_WT.csv')
# Create new columns for position and mutant amino acid
# Create new columns for position and mutant amino acid
df = pd.merge(activity,embeddings, on='seq_id')
def string_to_array(embedding_string):
    # Convert string to list
    embedding_list = ast.literal_eval(embedding_string)
    # Convert list to numpy array
    return np.array(embedding_list)

# Apply conversion to your dataframe
# If your embedding column is called 'embedding':
X = np.stack(df['embedding'].apply(string_to_array).values)

# Now proceed with dimensionality reduction
from sklearn.decomposition import PCA
# or from sklearn.manifold import TSNE
# or from umap import UMAP

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

# Create visualization as before
import plotly.express as px

embedding_df = pd.DataFrame({
    'x': X_2d[:, 0],
    'y': X_2d[:, 1],
    'activity': df['activity'],
    'seq_id': df['seq_id']
})

fig = px.scatter(embedding_df, 
                 x='x', 
                 y='y', 
                 color='activity',
                 hover_data=['seq_id'])
fig.show()

In [3]:
import pandas as pd
from evolve.evolvetest import plot_combined_landscape
activity = r'..\EvolveTest\DMS_data\brenan_VRT_dms.xlsx'
embeddings = r'..\EvolveTest\brenan\embeddings\brenan_esmc_600m_WT.csv'
logits = 'logits_brenan_wt_melted.csv'
protname = 'brenan_VRT'
plot_combined_landscape(activity,embeddings,logits,protname,embedding_dir=('../EvolveTest/' + 'brenan'))